# EdgentRAG model services on Google Colab

This notebook runs the embedding API on port 8001 and generation API on port 8003 in Colab, then gives each a temporary public HTTPS URL. Keep the notebook running while your local backend uses the URLs. These quick tunnels are for development, not production.

Before running, add `EDGENTRAG_EMBEDDING_API_TOKEN` and `EDGENTRAG_GENERATION_API_TOKEN` under Colab Secrets (key icon) and allow this notebook to read them. Keep the embedding token the same as the one configured in your local `.env`.

In [ ]:
from pathlib import Path
import json, os, platform, re, socket, subprocess, sys, time, urllib.error, urllib.request

if sys.version_info < (3, 12):
    raise RuntimeError('Use a Colab runtime with Python 3.12 or newer.')
from google.colab import userdata

PROJECT_DIR = Path('/content/ai-eng')
if not (PROJECT_DIR / 'app/backend/src/edgentrag/embedding/app.py').is_file():
    repo_url = input('Public HTTPS Git clone URL for this project: ').strip()
    if not repo_url.startswith('https://') or '@' in repo_url:
        raise ValueError('Enter a public HTTPS repository URL without credentials.')
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(PROJECT_DIR)], check=True)
for service in ('embedding', 'generation'):
    if not (PROJECT_DIR / f'app/backend/src/edgentrag/{service}/app.py').is_file():
        raise RuntimeError(f'Missing {service} API in {PROJECT_DIR}.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{PROJECT_DIR}/app/backend[embedding,generation]'], check=True)

for service in ('embedding', 'generation'):
    key = f'EDGENTRAG_{service.upper()}_API_TOKEN'
    try:
        value = userdata.get(key)
    except Exception:
        raise RuntimeError(f'Add {key} to Colab Secrets and enable notebook access.') from None
    if not value or not value.strip():
        raise RuntimeError(f'{key} cannot be empty.')
    os.environ[key] = value
os.environ['EDGENTRAG_EMBEDDING_DEVICE'] = 'cpu'
os.environ['EDGENTRAG_GENERATION_DEVICE'] = 'auto'
os.environ['EDGENTRAG_EMBEDDING_MODEL_NAME'] = 'sentence-transformers/all-MiniLM-L6-v2'
os.environ['EDGENTRAG_GENERATION_MODEL_NAME'] = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
print('Project installed; both API tokens are available (values hidden).')

## Start and warm the APIs

The first authenticated call downloads and loads model weights, so it may take several minutes. Embeddings use CPU by default to leave GPU memory for generation.

In [ ]:
model_processes = globals().get('model_processes', {})
model_tunnels = globals().get('model_tunnels', {})
def stop_process(process):
    if process.poll() is None:
        process.terminate()
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait(timeout=5)
def wait_health(url, process, log_path, attempts=60):
    for _ in range(attempts):
        if process.poll() is not None: raise RuntimeError(f'Process exited; inspect {log_path}.')
        try:
            with urllib.request.urlopen(url + '/health', timeout=5) as response: return json.load(response)
        except (urllib.error.URLError, TimeoutError): time.sleep(1)
    raise RuntimeError(f'Health check timed out; inspect {log_path}.')
def post_model(name, base_url):
    if name == 'embedding': route, payload = '/embed', {'texts':['A test document chunk.']}
    else: route, payload = '/generate', {'prompt':'Explain semantic search in one sentence.','max_new_tokens':64}
    request = urllib.request.Request(base_url + route, data=json.dumps(payload).encode(), headers={'Authorization':'Bearer '+os.environ[f'EDGENTRAG_{name.upper()}_API_TOKEN'],'Content-Type':'application/json'}, method='POST')
    with urllib.request.urlopen(request, timeout=600) as response: return json.load(response)

for name, port in {'embedding':8001,'generation':8003}.items():
    old = model_processes.get(name)
    if old is not None and old.poll() is None: raise RuntimeError(f'{name} already runs; clean up before rerunning.')
    with socket.socket() as probe:
        try: probe.bind(('127.0.0.1', port))
        except OSError: raise RuntimeError(f'Port {port} is occupied.') from None
started = []
try:
    for name, port in {'embedding':8001,'generation':8003}.items():
        log_path = Path(f'/tmp/edgentrag-{name}.log')
        env = os.environ.copy()
        env.pop(f"EDGENTRAG_{'GENERATION' if name == 'embedding' else 'EMBEDDING'}_API_TOKEN", None)
        with log_path.open('w') as log:
            process = subprocess.Popen([sys.executable,'-m','uvicorn',f'edgentrag.{name}.app:app','--host','127.0.0.1','--port',str(port)], cwd=PROJECT_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
        model_processes[name] = process; started.append(process)
        print(name, wait_health(f'http://127.0.0.1:{port}', process, log_path))
except Exception:
    for process in reversed(started): stop_process(process)
    raise

for name, port in {'embedding':8001,'generation':8003}.items():
    result = post_model(name, f'http://127.0.0.1:{port}')
    print(name, 'ready:', result.get('model', 'response received'))

## Create public URLs and verify access

Cloudflare Quick Tunnels make temporary HTTPS URLs without an account. The URLs change when the tunnel restarts. Each URL below is tested with the service's bearer token, which confirms it is usable by an external API client.

In [ ]:
architecture = {'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
if not architecture: raise RuntimeError('Unsupported Colab CPU architecture.')
package_path = Path('/tmp/edgentrag-cloudflared.deb')
urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-'+architecture+'.deb', package_path)
subprocess.run(['dpkg','-i',str(package_path)], check=True)
MODEL_URLS = {}
started_tunnels = []
try:
    for name, port in {'embedding':8001,'generation':8003}.items():
        log_path = Path(f'/tmp/edgentrag-{name}-tunnel.log')
        env = os.environ.copy(); env.pop('EDGENTRAG_EMBEDDING_API_TOKEN',None); env.pop('EDGENTRAG_GENERATION_API_TOKEN',None)
        with log_path.open('w') as log:
            process = subprocess.Popen(['cloudflared','tunnel','--url',f'http://127.0.0.1:{port}'],env=env,stdout=log,stderr=subprocess.STDOUT)
        model_tunnels[name] = process; started_tunnels.append(process)
        for _ in range(60):
            if process.poll() is not None: break
            match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log_path.read_text())
            if match: MODEL_URLS[name] = match.group(0); break
            time.sleep(1)
        if name not in MODEL_URLS: raise RuntimeError(f'No {name} tunnel URL; inspect {log_path}.')
        wait_health(MODEL_URLS[name], process, log_path)
        result = post_model(name, MODEL_URLS[name])
        print(name, 'public API verified:', result.get('model','response received'))
except Exception:
    for process in reversed(started_tunnels): stop_process(process)
    raise

print('EDGENTRAG_USE_COLAB_FOR_EMBEDDING=true')
print('EDGENTRAG_USE_COLAB_FOR_LLM=true')
print('EDGENTRAG_COLAB_EMBEDDING_SERVICE_URL=' + MODEL_URLS['embedding'])
print('EDGENTRAG_COLAB_GENERATION_SERVICE_URL=' + MODEL_URLS['generation'])

## Configure the local backend

Copy the printed URL settings into your local `.env`, keep `EDGENTRAG_EMBEDDING_API_TOKEN` equal to the token saved in Colab, and set the same generation token if you call generation directly. Restart the local API and worker after changing the embedding URL. Generation is not yet connected to search.

Optional cleanup: run the cell below with `STOP_SERVICES = True` to stop these API and tunnel processes. The Colab runtime must remain active while the URLs are used.

In [ ]:
STOP_SERVICES = False
if STOP_SERVICES:
    for registry in (model_tunnels, model_processes):
        for process in registry.values(): stop_process(process)
    MODEL_URLS.clear()
    print('Model APIs and tunnels stopped.')
else:
    print('Services remain running.')